<a href="https://colab.research.google.com/github/youssefabozaidyou/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
!pip -q install duckdb datasets pyarrow

In [16]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Connected successfully!")

Connected successfully!


In [17]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{DATASET}/fact_content_daily_performance/**/*.parquet'
)
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of analysis

One row represents the daily performance of one content item for one client.

Grain:
- report_date
- client_hash_id
- content_hash_id

Time window:
This notebook uses the month = '2026-03' as the analysis window, following the assignment recommendation to use a mid-panel month instead of the final month.

In [18]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    '{DATASET}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features

- gsc_impressions
- gsc_avg_position
- ga4_sessions
- ga4_users
- scroll_events

## Label

- gsc_clicks (proxy target)

## Context

- report_date
- client_hash_id
- content_hash_id
- month

## Excluded

- client_hash_id
  Reason: identifier only, not useful as a predictive feature.

- content_hash_id
  Reason: identifier only and may lead to memorization rather than generalization.

### Fix: context vs. excluded (from feedback)

`client_hash_id` and `content_hash_id` belong in **Context only** (used for grouping/joining/
splitting, never as model inputs) -- they should not also appear under **Excluded**. Excluded is
reserved for fields dropped for a specific reason, such as `trend_direction` below.

### Five features, each with a "knowable at decision moment" line

1. `gsc_impressions_avg_prior` -- avg impressions over days *before* the decision point -- knowable
   because it only uses past days, nothing from the day/window being predicted.
2. `gsc_avg_position_prior` -- avg search position over prior days -- knowable, purely historical.
3. `ga4_sessions_avg_prior` -- avg GA4 sessions over prior days (only when `ga4_data_available IS TRUE`)
   -- knowable, historical, and correctly guarded against the zero-fill trap.
4. `ga4_users_avg_prior` -- avg GA4 users over prior days -- same reasoning as above.
5. `n_days_present_prior` -- count of days the content appeared in the prior window -- knowable,
   counts only past days.

**Excluded (with reason):**
- `trend_direction` -- derived from the same trend the label is built from; used deliberately in
  the leakage trap below to show why it can never be a feature.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [19]:
con.sql(f"""
SELECT
report_date,
client_hash_id,
content_hash_id,
COUNT(*) AS n
FROM read_parquet(
'{DATASET}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
GROUP BY
report_date,
client_hash_id,
content_hash_id
HAVING COUNT(*)>1
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,n


In [21]:
con.sql(f"""
SELECT
COUNT(*) "rows",
MIN(report_date) start_date,
MAX(report_date) end_date
FROM read_parquet(
'{DATASET}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
""").df()

,rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [22]:
con.sql(f"""
SELECT
COUNT(*) available_rows
FROM read_parquet(
'{DATASET}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
AND gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


In [23]:
# Missing-values check (section 3 requirement) -- per key column, on the same slice
con.sql(f"""
SELECT
    AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END)        AS pct_null_clicks,
    AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END)   AS pct_null_impressions,
    AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END)  AS pct_null_position,
    AVG(CASE WHEN gsc_avg_position = 0 THEN 1.0 ELSE 0 END)      AS pct_zero_position_no_data,
    AVG(CASE WHEN gsc_data_available IS NOT TRUE THEN 1.0 ELSE 0 END) AS pct_gsc_unavailable
FROM read_parquet(
    '{DATASET}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,pct_null_clicks,pct_null_impressions,pct_null_position,pct_zero_position_no_data,pct_gsc_unavailable
0,0.0,0.0,0.633074,0.016582,0.633074


## The trap: deliberate leakage experiment (missing piece added)

Train an honest model on the 5 features above, then deliberately add a column derived from the
label itself, watch the score jump toward perfect, then remove it and keep the honest number.

**Note on label choice:** the earlier `gsc_clicks` label used same-row `gsc_impressions` /
`gsc_avg_position` as features -- same day as the label, so there's no real prediction gap. The
label below (`clicks_will_rise_next_period`) is built from a later window than the features, to
give an honest before/after split.


In [24]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

daily = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, gsc_clicks, gsc_impressions, gsc_avg_position
    FROM read_parquet('{DATASET}/fact_content_daily_performance/**/*.parquet')
    WHERE month='2026-03' AND gsc_avg_position > 0
""").df()
daily["report_date"] = pd.to_datetime(daily["report_date"])

# Simplified before/after split within the month (prior half -> features, later half -> label)
cutoff = daily["report_date"].quantile(0.5)
prior = daily[daily["report_date"] <= cutoff]
later = daily[daily["report_date"] > cutoff]

feat = prior.groupby(["client_hash_id", "content_hash_id"]).agg(
    gsc_impressions_avg_prior=("gsc_impressions", "mean"),
    gsc_avg_position_prior=("gsc_avg_position", "mean"),
    n_days_present_prior=("report_date", "nunique"),
    clicks_avg_prior=("gsc_clicks", "mean"),
).reset_index()

label_src = later.groupby(["client_hash_id", "content_hash_id"]).agg(
    clicks_avg_later=("gsc_clicks", "mean")
).reset_index()

data = feat.merge(label_src, on=["client_hash_id", "content_hash_id"], how="inner")
data["clicks_will_rise_next_period"] = (data["clicks_avg_later"] > data["clicks_avg_prior"]).astype(int)

feature_cols = ["gsc_impressions_avg_prior", "gsc_avg_position_prior",
                 "n_days_present_prior", "clicks_avg_prior"]

X = data[feature_cols].fillna(0)
y = data["clicks_will_rise_next_period"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

model_honest = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_score = accuracy_score(y_test, model_honest.predict(X_test))
print("Honest accuracy (no leakage):", round(honest_score, 3))

# Deliberately add a leaked column derived from the label itself
data["trend_direction_LEAK"] = data["clicks_will_rise_next_period"]  # exact same information

X_leak = data[feature_cols + ["trend_direction_LEAK"]].fillna(0)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leak, y, test_size=0.25, random_state=42, stratify=y
)

model_leak = LogisticRegression(max_iter=1000).fit(X_train_l, y_train_l)
leak_score = accuracy_score(y_test_l, model_leak.predict(X_test_l))
print("Accuracy with the leak column:", round(leak_score, 3))
print("-> Jumps close to 1.0 -- trend_direction_LEAK is derived from the label, not a real feature.")

# Remove the leaked column, keep the honest number
print("\nFinal honest number (leak column removed):", round(honest_score, 3))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest accuracy (no leakage): 0.796
Accuracy with the leak column: 1.0
-> Jumps close to 1.0 -- trend_direction_LEAK is derived from the label, not a real feature.

Final honest number (leak column removed): 0.796


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data limits

- The dataset is an unbalanced panel, so clients have different history lengths.
- Some clients have only GSC or only GA4 data.
- This dataset cannot identify real clients because all identifiers are pseudonymized.
- Results should be interpreted as decision-support rather than causal conclusions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.